# 02_benchmark

Latency and quality benchmarking of DeclipNet across runtimes and precisions.

1. **Load models** — PyTorch (MPS), ONNX Runtime (CPU), Core ML FP32 (ANE)
2. **INT8 post-training quantization** — `coremltools.optimize.coreml.linear_quantize_weights` on the FP32 `.mlpackage`; export quantized Core ML INT8
3. **Benchmark all variants** — per-block latency, per-utterance OLA latency, SI-SDR, PESQ, and model file size for all four: PyTorch MPS, ONNX CPU, Core ML FP32, Core ML INT8
4. **Summary table** — variant × precision → latency, SI-SDR, PESQ, file size

**Prerequisites**
- `models/declipnet.onnx` and `models/declipnet.mlpackage` from `02_export`
- Pruning evaluated in `02_pruning` — not viable for this architecture (quality collapsed at all sparsity levels)

**Caveat**
- Core ML latency is measured via `coremltools` Python `predict()`, which adds overhead versus native Swift `MLModel.prediction()`. Absolute numbers are a soft upper bound on native app performance; relative FP32 vs INT8 comparison is still valid since both use the same path.

In [1]:
import sys
import time
import json
import torch
import numpy as np
import onnxruntime as ort
import coremltools as ct
from pathlib import Path

sys.path.insert(0, "..")
sys.path.insert(0, "../01_dnn")

from config import *
from util import DeclipNet
from deploy_util import eval_ola_routed

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

H = 8
N_ATTN = 3
NUM_HEADS = 4
FFN_DIM = 256

MODELS_DIR = Path("models")

ONNX_PATH = MODELS_DIR / "declipnet.onnx"
CML_FP32_PATH = MODELS_DIR / "declipnet.mlpackage"
CML_INT8_PATH = MODELS_DIR / "declipnet_int8.mlpackage"

scikit-learn version 1.7.2 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.11.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.


Device: mps


In [2]:
# Load PyTorch model
model = DeclipNet(H=H, N=N_ATTN, num_heads=NUM_HEADS, ffn_dim=FFN_DIM)
state_dict = torch.load(
    FINAL_OUT / "weighted_l1_dwt" / "best_model.pt",
    weights_only=True, map_location="cpu",
)
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.to(DEVICE).eval()
print(f"PyTorch DeclipNet — {sum(p.numel() for p in model.parameters()):,} params on {DEVICE}")

# Load ONNX Runtime session
ort_sess = ort.InferenceSession(str(ONNX_PATH))
print(f"ONNX Runtime session — {ONNX_PATH.name}")

# Load Core ML FP32
cml_fp32 = ct.models.MLModel(str(CML_FP32_PATH))
print(f"Core ML FP32 — {CML_FP32_PATH.name}")

# Load test manifest
with open(TEST_OUT / "test_manifest.json") as f:
    test_manifest = json.load(f)

PyTorch DeclipNet — 314,001 params on mps
ONNX Runtime session — declipnet.onnx
Core ML FP32 — declipnet.mlpackage


In [3]:
# INT8 post-training quantization of Core ML model
import coremltools.optimize.coreml as cto_coreml

quant_config = cto_coreml.OptimizationConfig(
    global_config=cto_coreml.OpLinearQuantizerConfig(
        mode="linear_symmetric",
        dtype="int8",
    )
)
cml_int8 = cto_coreml.linear_quantize_weights(cml_fp32, quant_config)
cml_int8.save(str(CML_INT8_PATH))
print(f"Core ML INT8 saved to {CML_INT8_PATH}")

# File size comparison
fp32_size = sum(f.stat().st_size for f in CML_FP32_PATH.rglob("*") if f.is_file())
int8_size = sum(f.stat().st_size for f in CML_INT8_PATH.rglob("*") if f.is_file())
print(f"\nFP32: {fp32_size / 1024:.1f} KB")
print(f"INT8: {int8_size / 1024:.1f} KB")
print(f"Reduction: {(1 - int8_size / fp32_size) * 100:.1f}%")

Running compression pass linear_quantize_weights: 100%|█| 71/71 [00:00<00:00,
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running MIL default pipeline: 100%|████| 92/92 [00:00<00:00, 234.30 passes/s]
Running MIL backend_mlprogram pipeline: 100%|█| 12/12 [00:00<00:00, 277.83 pa


Core ML INT8 saved to models/declipnet_int8.mlpackage

FP32: 691.2 KB
INT8: 407.8 KB
Reduction: 41.0%


In [5]:
from util import si_sdr
from pesq import pesq as pesq_fn


class OnnxInfer:
    """Wraps ORT session to match PyTorch callable interface."""
    def __init__(self, session):
        self.session = session
    def __call__(self, x):
        out = self.session.run(None, {"clipped": x.numpy()})[0]
        return torch.from_numpy(out)


class CoreMLInfer:
    """Wraps Core ML model (batch=1 loop) to match PyTorch callable interface."""
    def __init__(self, mlmodel):
        self.mlmodel = mlmodel
    def __call__(self, x):
        out = [self.mlmodel.predict({"clipped": x[i:i+1].numpy()})["restored"]
               for i in range(x.shape[0])]
        return torch.from_numpy(np.concatenate(out))


variants = [
    ("PyTorch MPS", model,                  DEVICE),
    ("ONNX CPU",    OnnxInfer(ort_sess),    torch.device("cpu")),
    ("CoreML FP32", CoreMLInfer(cml_fp32),  torch.device("cpu")),
    ("CoreML INT8", CoreMLInfer(cml_int8),  torch.device("cpu")),
]

# --- Per-block latency ---
N_WARMUP = 10
N_BENCH = 100
results = {}

print("Per-block latency (single block, batch=1):\n")
for name, m, dev in variants:
    x = torch.randn(1, 1, BS, device=dev)
    with torch.no_grad():
        for _ in range(N_WARMUP):
            m(x)
        if dev.type == "mps":
            torch.mps.synchronize()
        t0 = time.perf_counter()
        for _ in range(N_BENCH):
            m(x)
        if dev.type == "mps":
            torch.mps.synchronize()
    block_ms = (time.perf_counter() - t0) / N_BENCH * 1000
    results[name] = {"block_ms": block_ms}
    print(f"  {name:15s} {block_ms:.3f} ms")

# --- OLA evaluation (latency + quality) on full test set ---
print(f"\nOLA + routed evaluation ({len(test_manifest)} utterances):\n")
for name, m, dev in variants:
    t0 = time.perf_counter()
    res = eval_ola_routed(m, test_manifest, dev, verbose=False)
    ola_s = time.perf_counter() - t0
    results[name].update({
        "ola_s": ola_s,
        "si_sdr": res["mean_sdr"],
        "pesq": res["mean_pesq"],
    })
    print(f"  {name:15s} {ola_s:6.1f}s | SI-SDR {res['mean_sdr']:.2f} dB | PESQ {res['mean_pesq']:.3f}")

# --- File sizes ---
pt_size = (FINAL_OUT / "weighted_l1_dwt" / "best_model.pt").stat().st_size
onnx_size = ONNX_PATH.stat().st_size
fp32_size = sum(f.stat().st_size for f in CML_FP32_PATH.rglob("*") if f.is_file())
int8_size = sum(f.stat().st_size for f in CML_INT8_PATH.rglob("*") if f.is_file())

results["PyTorch MPS"]["size_kb"] = pt_size / 1024
results["ONNX CPU"]["size_kb"] = onnx_size / 1024
results["CoreML FP32"]["size_kb"] = fp32_size / 1024
results["CoreML INT8"]["size_kb"] = int8_size / 1024

Per-block latency (single block, batch=1):

  PyTorch MPS     2.237 ms
  ONNX CPU        0.663 ms
  CoreML FP32     0.480 ms
  CoreML INT8     0.477 ms

OLA + routed evaluation (100 utterances):

  PyTorch MPS       41.6s | SI-SDR 27.32 dB | PESQ 3.946
  ONNX CPU           8.6s | SI-SDR 27.32 dB | PESQ 3.946
  CoreML FP32        8.5s | SI-SDR 27.26 dB | PESQ 3.940
  CoreML INT8        8.4s | SI-SDR 27.24 dB | PESQ 3.939


In [6]:
import pandas as pd

rows = []
for name, metrics in results.items():
    rows.append({
        "Variant": name,
        "Block (ms)": f"{metrics['block_ms']:.3f}",
        "OLA (s)": f"{metrics['ola_s']:.1f}",
        "SI-SDR (dB)": f"{metrics['si_sdr']:.2f}",
        "PESQ": f"{metrics['pesq']:.3f}",
        "Size (KB)": f"{metrics['size_kb']:.1f}",
    })

df = pd.DataFrame(rows).set_index("Variant")
df

,Block (ms),OLA (s),SI-SDR (dB),PESQ,Size (KB)
Variant,,,,,
PyTorch MPS,2.237,41.6,27.32,3.946,1253.5
ONNX CPU,0.663,8.6,27.32,3.946,1288.8
CoreML FP32,0.480,8.5,27.26,3.940,691.2
CoreML INT8,0.477,8.4,27.24,3.939,407.8


## Summary

- **Latency**: PyTorch MPS ~5x slower than all others
   - GPU kernel launch overhead for single batch, 314k params)
   - ONNX CPU and both CoreML variants clustered at ~0.5 ms/block
- **Quality**: all four variants within 0.08 dB SI-SDR and 0.007 PESQ - no meaningful degradation from export or quantization
- **FP32 vs INT8 CoreML**: wash — weights are ~600 KB fp16, fit in cache, INT8 does not provide compute gains (only saves ~280 KB on disk)
- **File size**: CoreML "FP32" stores weights as fp16 internally (ANE path), hence ~691 KB vs ~1.3 MB for PyTorch/ONNX float32; INT8 reduces to ~408 KB (41%)
- **Recommendation**: CoreML FP32 `.mlpackage` is the deployment pick — fastest tier, smallest fp format, no quantization risk